In [18]:
# ============================================================
# SETUP BATCH EVENTS — LOCAL EXECUTION
# ============================================================

import os, shutil, uuid
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [19]:
# -----------------------------------------
# 1. Initialize Spark Session
# -----------------------------------------
spark = (
    SparkSession.builder
        .appName("F1-Analytics-Batch-Events")
        .master("local[*]")
        .getOrCreate()
)

In [20]:
# -----------------------------------------
# 2. Define Control Table Path
# -----------------------------------------
CONTROL_PATH = "/Users/manoharazalki/F1-Analytics/data/control"
BATCH_EVENTS_PATH = f"{CONTROL_PATH}/batch_events.parquet"

os.makedirs(CONTROL_PATH, exist_ok=True)

In [21]:
# -----------------------------------------
# 3. Create Batch Events Table (Local Parquet)
# -----------------------------------------
from pyspark.sql.types import StructType, StructField, IntegerType, TimestampType

schema = StructType([
    StructField("batch_id", IntegerType(), True),
    StructField("event_timestamp", TimestampType(), True)
])

# Create empty table if not exists
if not os.path.exists(BATCH_EVENTS_PATH):
    empty_df = spark.createDataFrame([], schema)
    empty_df.write.mode("overwrite").parquet(BATCH_EVENTS_PATH)
    print("✔ Created batch_events table.")
else:
    print("✔ batch_events table already exists.")

✔ batch_events table already exists.


In [22]:
# -----------------------------------------
# 4. Insert Initial Batch Event
# -----------------------------------------
CONTROL_PATH = "/Users/manoharazalki/F1-Analytics/data/control"
BATCH_EVENTS_PATH = f"{CONTROL_PATH}/batch_events.parquet"

def parquet_exists(path):
    return os.path.exists(path) and any(
        f.endswith(".parquet") for f in os.listdir(path)
    )

# Create initial record
initial_df = spark.createDataFrame(
    [(1,)], 
    ["batch_id"]
).withColumn("event_timestamp", F.current_timestamp())

# Read existing if valid parquet exists
if parquet_exists(BATCH_EVENTS_PATH):
    existing_df = spark.read.parquet(BATCH_EVENTS_PATH)
    final_df = existing_df.unionByName(initial_df)
else:
    final_df = initial_df

# Safe overwrite using temp folder
temp_path = f"{CONTROL_PATH}/_tmp_{uuid.uuid4().hex}"
final_df.write.mode("overwrite").parquet(temp_path)

# Remove old folder if exists
if os.path.exists(BATCH_EVENTS_PATH):
    shutil.rmtree(BATCH_EVENTS_PATH)

# Move temp → final
shutil.move(temp_path, BATCH_EVENTS_PATH)

print("✔ Batch event inserted successfully.")

✔ Batch event inserted successfully.


In [23]:
# -----------------------------------------
# 5. Display Batch Events
# -----------------------------------------
display(spark.read.parquet(BATCH_EVENTS_PATH))


DataFrame[batch_id: bigint, event_timestamp: timestamp]